# 07 · Deposition and optical depth

This notebook goes from the optical-depth integral along a path to the sum over pieces that SkewerJAX evaluates, and
from the gas cells to $\tau$, on one skewer through the TNG50-4 snapshot of notebook 05, with the package's own code
path. The derivation comes first, below; then:

1. the deposition, step by step on that skewer: selection, deposition into pieces, the sum over the pieces, and
   periodic boundaries;
2. the line-centred frames, the global velocity axis and the observed wavelength, for $\mathrm{Ly}\alpha$ and Si III on
   the same skewer.

The profile that every piece adds, that of a gas element, is derived in notebook 06. The closed-form checks of $\tau$
(static gas with natural or thermal widths, radiation-dominated expansion) are in
`scripts/validation/check_optical_depth_analytic.py`, run by the test suite. Runtime: ~1 min.

### From the integral to the sum over pieces

**The integral.** Along a straight comoving path, a photon is absorbed by transition $j$ of species $i$ with optical
depth (review notes, Eq. `tau_straight_los`, with $dl_{\rm phys} = a\, dx$)

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^3}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, a(x)\, dx
= \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^2}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, dx,
\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c} .
$$

It is a line integral along the path of the light. At every $x$ the gas adds its line profile $\phi_j$: the Voigt
function at the detuning $\nu_j$ of the photon from the line centre $v_c(x)$ of the gas, with the widths $\sigma$ and
$\gamma$ of its temperature and transition (notebook 06, §2).

**Step 1: cut the path into pieces.** The integral is the sum of the integrals over the parts of any partition of the
path. SkewerJAX uses **pieces** $p$, of length $\ell_p$: fragments of the path within which the gas properties are
constant. Every source gives the gas as Voronoi cells (`core.Cells`: the toy models of 04, the snapshots of 05), uniform
inside each cell, so the pieces are the exact intersections of the path with the cells (§1). By default they are also
cut at the ends of the dual segments of the sampling points, one piece per cell and dual segment; the cell crossings
(`deposition="voronoi_crossings"`) keep one interval per crossing of a cell, split into equal sub-pieces.

**Step 2: what is constant within a piece, and what is not.** Constant, exactly, because a piece lies in one cell:
$n_{\rm com}$, $T$ (and so $\sigma$ and $\gamma$) and the velocity vector; the path is straight within a piece, so
$v_\parallel = \mathbf{v}\cdot\hat t$ is constant too. Not constant: $a(x)$, so $a^{-2}$ and the Hubble part of the line
centre, $v_H(x) = c\ln(a_\dagger/a(x))$, change across every piece. The line centre moves across a piece by

$$
W_p = \left\lvert\frac{dv_c}{dx}\right\rvert \ell_p = \left\lvert\frac{dv_H}{dx}\right\rvert \ell_p ,
$$

the Hubble rate times the length of the piece.

**Step 3: one profile per piece, at its centroid.** The scale factor $a$, $a^{-2}$ and $v_c$ are evaluated at the
centroid $s_p$ of every piece ($a$ interpolated in $\ln a$ between the bundle's points, where $a(x)$ was solved), with
the values of its cell. This is the midpoint rule on each piece:

$$
\tau_j(v_\dagger) \simeq \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\, \ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \phi_{j,p}(v_\dagger) = V\big(\nu_j(v_\dagger - v_{c,p});\, \sigma_p, \gamma_p\big),
$$

with the widths $\sigma_p$, $\gamma_p$ of the piece's cell and the exact detuning of notebook 06, §2. This is
`core.optical_depth_kernel`, mapped over skewers by `optical_depth_from_pieces`, behind
`compute_optical_depth(source, v, lines, ...)`, whose source is a `SkewerBundle` with its `Cells`, or `Pieces`.

**Error, and when the sum is exact.** Both follow from the gas being piecewise constant:

- The midpoint rule replaces the profile smeared over $W_p$ by the profile at its centre. For a Gaussian core this
  changes the peak by a relative $W_p^2/(12 b^2)$: the sum converges to the integral at second order in the piece
  length, i.e. in the spacing of the sampling points.
- $\phi$ has unit area in $\nu$, so $\int \tau\, d\nu = \mathcal{C}_j \sum_p n_{{\rm com},p}\, \ell_p\, a_p^{-2}$ for any
  partition. The column $\sum_p n_p \ell_p$ is exact, for any spacing, because $n$ is exactly constant in a cell and the
  lengths are exact.
- The sum equals the integral exactly when $a$ and $v_c$ are constant within every piece; only the Hubble flow varies
  inside a cell. It converges to the integral as $\max_p W_p/b_p \to 0$.
- The resolution check of `compute_optical_depth` (`resolution_report`) monitors exactly this ratio, with the width of
  every piece (`Pieces.width_kms`). With the simulation default of 0.25 km/s spacing, $W_p \le 0.25$ km/s, far below
  any thermal width.

§1 builds this sum step by step, on a skewer through TNG50-4, and adds the one option that changes where the pieces
absorb: a periodic spectrum.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'simulation', 'fields', 'profiles', 'deposition', 'tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, SkewerBundle, compute_optical_depth, resolution_report, phys)
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")             # the two lines of this notebook
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## 1 · Deposition and periodic boundaries, step by step

Three different grids meet in a simulated spectrum, and the deposition is what connects them:

- **the cells:** one value per Voronoi cell, the region closer to its mesh-generating point than to any other;
- **the sampling points of the `SkewerBundle`:** point $i$ stands for its *dual segment*, between the midpoints with its
  neighbours, of length $w_i$ (03);
- **the spectral grid** of $v$ on which $\tau$ is evaluated, chosen freely.

What connects them are the **pieces**: the fragments of the path inside one cell, within which the gas is constant.
Every source of gas reaches $\tau$ as cells (`core.Cells`: the toy models of 04, the snapshots of 05), and from there
by one route. The steps below run that route, one function per step, on a real skewer; `compute_optical_depth(bundle,
v, lines, cells)` chains the same functions in one call, for a toy model or a snapshot alike. The last step makes the
spectrum periodic, an option of $\tau$.

**The skewer.** A straight line along $x$ through the centre of the TNG50-4 box ($y = z = L/2$), from one face to the
opposite one: a `SkewerBundle` with the snapshot's cosmology, its scale factor at the first point and the observer
before it. Its points are 8 km/s of Hubble flow apart (0.1 cMpc), about one per cell: coarse on purpose, so that the
pieces of the figure below are few and can be followed one by one (§2 uses the same line sampled every km/s). **The cells** are those of a slab of $\pm1.5$ cMpc around it, read as in
notebook 05 (`snapshot_cells`: the gas physics of 05, §2, per cell, for H I and Si III).

In [ ]:
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with TNG50-4/: $SKEWERJAX_DATA (docs/installation.md)
from skewerjax.simulations import ArepoHDF5Reader, snapshot_cells
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
L, yc = H.box_cMpc, H.box_cMpc / 2
cosmo = H.cosmology()                                     # the snapshot's flat LCDM
x = np.linspace(0.0, L, int(round(H.vmax_kms / 8.0)) + 1)                 # one point per 8 km/s of Hubble flow (coarse, for the figure); first = last place
skewer = SkewerBundle(spatial_points=jnp.asarray(np.stack([x, np.full_like(x, yc), np.full_like(x, yc)], -1)[None]), a0=H.a, x0=0.0,
                      ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)   # observer before the first point
gas = snapshot_cells(reader, ["HI", "SiIII"], [0.0, yc - 1.5, yc - 1.5], [L, yc + 1.5, yc + 1.5])   # the slab: core.Cells, periodic in x
print(f"one skewer of {skewer.N_ppsk:,} points along x at y = z = {yc:.2f} cMpc, {float(skewer.path_lengths[0, -1]):.2f} cMpc; "
      f"a from {float(skewer.scale_factors[0, 0, 0]):.5f} to {float(skewer.scale_factors[0, 0, -1]):.5f}")
print(f"{len(gas):,} cells in the slab around it, out of {H.n_gas:,}")

**1. Selection.** `select_cells` keeps the cells within reach of the points, as in notebook 05 (the same rule as the
streamed `select_cells_near_paths` of `SimulationSkewers`).

In [ ]:
from skewerjax.core import (select_cells, voronoi_segments, cell_crossings, absorbers_from_components, bundle_frame, absorber_pieces,
                            cell_pieces)
near = select_cells(gas, skewer.spatial_points)                              # within 2.5 cell sizes of a point
print(f"selection: {len(near):,} of {len(gas):,} cells can reach the skewer")

**2. Deposition.** `voronoi_segments` cuts every dual segment at the exact boundaries of the cells it crosses
(the lower envelope of the distances to the generators): one piece per (cell, dual segment). `cell_crossings` joins
the pieces of one cell across the midpoints, which leaves one interval per crossing of a cell.

In [ ]:
pts = skewer.spatial_points
seg = voronoi_segments(pts, near.positions, L, gas_size=near.size_cMpc)        # pieces of every dual segment
cr = cell_crossings(seg, pts)                                                 # the same pieces joined per cell
print(f"deposition: {int((seg.cell >= 0).sum()):,} pieces in {skewer.N_ppsk:,} dual segments; {int(cr.n_crossings[0])} cell crossings")
print(f"the pieces tile the path: sum of their lengths {seg.length[seg.cell >= 0].sum():.6f} = path length "
      f"{float(skewer.path_lengths[0, -1]):.6f} cMpc")

**3. Pieces.** `absorbers_from_components` gives every piece its centroid $s_p$, its cell's density and
temperature, and the projection of its cell's velocity on the path, $v_{\parallel,p} = \mathbf{v}\cdot\hat t$. The
frame of the bundle (`bundle_frame`) gives $a(s_p)$ along the path and the direction of the photons, and
`absorber_pieces` turns them into `Pieces`, what the sum runs over. `cell_pieces` runs the three steps for every
species of a `LineBundle` in one call, from the cells of the slab.

In [ ]:
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
ab = absorbers_from_components(seg, pts, near, "HI", float(lya.mass_amu[0]))   # s (centroid), n, l, T, v_par of every piece
frame = bundle_frame(skewer)                                                  # a(s) along the path, a_dagger = a(0), photon direction
pieces = absorber_pieces(ab, frame, lya.n_lines)                             # Pieces: what the tau sum runs over
same, parts = cell_pieces(gas, lya, skewer_bundle=skewer, return_parts=True)  # the three steps in one call (and their parts)
print(f"cell_pieces (selection + deposition + pieces in one call): the same pieces, max |dl| = "
      f"{np.abs(np.sort(np.asarray(same.length_m)) - np.sort(np.asarray(pieces.length_m))).max() / phys.Mpc_to_m:.1e} cMpc")
live = ab.n[0] > 0
print(f"{int(live.sum()):,} pieces in {len(np.unique(ab.cell[0][live]))} cells; v_par from {ab.v_los[0][live].min():+.0f} to "
      f"{ab.v_los[0][live].max():+.0f} km/s; Hubble widths W_p up to {float(pieces.width_kms.max()):.2f} km/s; "
      f"velocity span of the path {float(pieces.velocity_span_kms[0]):.1f} km/s")

**4. The optical depth.** `core.compute_optical_depth` sums one profile per piece,

$$
\tau(v) = \sum_p d\tau_p(v),\qquad d\tau_p(v) = \mathcal{C}_j\, n_{{\rm com},p}\, \ell_p\, a_p^{-2}\, \phi_{j,p}(v),
$$

centred at the line centre of its centroid, $v_{c,p} \simeq c\ln(a_\dagger/a(s_p)) + v_{\parallel,p}$ (the derivation
above gives this sum and its accuracy). `per_piece=True` returns every $d\tau_p$. With a point every 8 km/s the pieces are up
to 8 km/s wide in Hubble flow, a large fraction of the thermal width, so the resolution check of `compute_optical_depth`
warns (below): the coarse points are for the figure, and the same line sampled every km/s (`skewer_fine`, used in §2)
passes the check.

In [ ]:
P = float(pieces.velocity_span_kms[0])                                     # the velocity span of the path, c ln(a_first / a_last)
v = np.arange(-300.0, P + 300.0, 1.0)                                      # 1 km/s pixels over the path and 300 km/s beyond each end
tau = np.asarray(compute_optical_depth(pieces, v, lya))[0, 0]              # the resolution check warns: coarse points
rep = resolution_report(pieces, lya)
print(f"tau = sum over {int(live.sum()):,} pieces: max {tau.max():.1f}, <F> on [0, P) = {np.exp(-tau[(v >= 0) & (v < P)]).mean():.3f}; "
      f"resolution check: max W/b = {rep['max_ratio'][0]:.2f}, column in pieces wider than b/4: {rep['column_fraction'][0]:.0%}")
x1 = np.linspace(0.0, L, int(round(H.vmax_kms)) + 1)                         # the same line, one point per km/s
skewer_fine = SkewerBundle(spatial_points=jnp.asarray(np.stack([x1, np.full_like(x1, yc), np.full_like(x1, yc)], -1)[None]), a0=H.a, x0=0.0,
                           ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
tau_fine = np.asarray(compute_optical_depth(skewer_fine, v, lya, gas))[0, 0]   # selection, deposition, pieces and tau in one call
print(f"sampled every km/s instead ({skewer_fine.N_ppsk:,} points): the check passes; max |dF| = {np.abs(np.exp(-tau_fine) - np.exp(-tau)).max():.2f}, "
      f"<F> = {np.exp(-tau_fine[(v >= 0) & (v < P)]).mean():.3f}")

**5. Periodic boundaries.** The cells are periodic: the gas past a face of the box is the gas at the opposite face, and
the selection and the deposition use its nearest images. That is the topology of the gas, not an option. The spectrum
is not periodic by nature. The light that crosses the path once is absorbed by the Hubble flow from $v_H = 0$ at one end
to $v_H = P = c\ln(a_{\rm first}/a_{\rm last})$ at the other, so the two ends map to different velocities, and what falls
beyond them is not on the spectrum. A periodic spectrum is a choice, `velocity_period` of `compute_optical_depth`:

- `False` (the default): every piece absorbs where its line centre is;
- `True`: each skewer is periodic over its own velocity span $P$, set by the scale factors at its first and last points
  (`Pieces.velocity_span_kms`);
- a number: that period [km/s] for every skewer.

**How the code wraps: one operation, on the detuning.** For every piece $p$, the
detuning $v - v_{c,p}$ is replaced by its nearest image modulo $P$, in $[-P/2, P/2)$ (`spectral.frequency_offset_GHz`).
The line centres themselves are never moved, nothing is computed outside the window.
So each piece's whole profile, centre and tails together, is laid on a circle of circumference $P$: a profile
near an edge is split, the part on one side stays where it is and the part beyond the edge is evaluated $P$ away, at the
other end. Two consequences are visible (rows (d) and (e) of the figure below):

- **(i)** a piece whose line centre falls outside the window absorbs at the other
  end, at $v_c \mp P$, apart from whatever part of its profile still lies inside;
- **(ii)** a piece whose line centre is inside, but whose profile reaches past an edge, has that tail re-enter at the
  other end.

**The cut at $P/2$.** Each pixel sees only the nearest image of each piece: the profile is cut half a period from its
centre, on the ray opposite it on the ring (the insets of rows (d) and (e)), and the other images are not summed. The exact periodic spectrum is the sum over all images,
$\sum_n \tau_1(v_\dagger + nP)$; the difference matters only for damped wings in small boxes.

In [ ]:
from skewerjax.core.spectral import line_centre_velocity_kms
tau_wrap = np.asarray(compute_optical_depth(pieces, v, lya, velocity_period=True))[0, 0]   # on the grid only; outside [0, P) it repeats
inside = (v >= 0) & (v < P)
n_px = int(np.ceil(P)); dv_w = P / n_px
v_win = (np.arange(n_px) + 0.5) * dv_w                                       # pixels that tile the window [0, P) exactly
per_k = [np.asarray(compute_optical_depth(pieces, v_win + k * P, lya, per_piece=True, resolution_tolerance=None))[0, 0]
         for k in (-1, 0, 1)]                                                # every piece, no wrap, on the windows below, at and above [0, P)
column = sum(t.sum() for t in per_k) * dv_w                                  # the integral of tau dv over [-P, 2P): the whole column
t_in, t_out = per_k[1].sum(0), per_k[0].sum(0) + per_k[2].sum(0)             # no wrap: inside the window, and P below / above it
t_wrap = np.asarray(compute_optical_depth(pieces, v_win, lya, velocity_period=True, resolution_tolerance=None))[0, 0]
v_c = np.asarray(line_centre_velocity_kms(pieces.a_dagger[0], pieces.scale_factor[0], pieces.los_velocity[0]))[0]
centre_out = live & ((v_c < 0) | (v_c >= P))
tails_ii = (per_k[0] + per_k[2])[live & ~centre_out].sum() * dv_w           # (ii): tails of centre-inside pieces beyond an edge
cut = ((t_in + t_out) - t_wrap).sum() * dv_w                                 # all images summed minus the nearest image
print(f"P = {P:.1f} km/s. Without wrap the window holds {t_in.sum() * dv_w / column:.4f} of the column: "
      f"{per_k[2].sum() * dv_w / column:.2%} falls beyond P, {per_k[0].sum() * dv_w / column:.1e} below 0")
print(f"(i) {int(centre_out.sum())} pieces have their line centre outside the window ({(per_k[0] + per_k[1] + per_k[2])[centre_out].sum() * dv_w / column:.2%} "
      f"of the column); (ii) the tails of centre-inside pieces across an edge hold {tails_ii / column:.1e} of it")
print(f"with velocity_period=True the window holds {t_wrap.sum() * dv_w / column:.6f} of the column; the cut at P/2 (images not summed) "
      f"removes {cut / column:.1e}")
print(f"balance: the no-wrap tau outside the window, {t_out.sum() * dv_w:.2f} km/s, = the gain inside it, "
      f"{(t_wrap - t_in).sum() * dv_w:.2f} km/s, + the cut, {cut:.2e} km/s")

**The figure.** A crossing of the box passes through hundreds of cells, and periodicity acts where the path closes on
itself. So the figure follows the two ends of the skewer, its last and its first 2 cMpc (`REGIONS`), **the end on the
left and the start on the right**: in every row the seam ($s = 0 \equiv L$ in space, $v_\dagger = 0 \equiv P$ in velocity) is
in the middle. Every cell keeps one colour, its H I density, and one label (S1, S2, ... at the start; E1, E2, ... at the
end). From row (b) on, the pieces are the objects: each is labelled by its cell and a letter in path order (E3a, E3b,
...). Rows (c)–(e) put the same pieces on rings, the natural geometry of a periodic coordinate: one angular window
around the seam (12 o'clock), the same in the three rings, and on the left the whole ring for orientation. Every
quantity comes from the objects above (`parts` of `cell_pieces`, and every $d\tau_p$ of
`compute_optical_depth(..., per_piece=True, velocity_period=True)` on one period).

In [ ]:
from skewerjax.utils.plotting import plot_deposition_steps
REGIONS = ((0.0, 2.0), (L - 2.0, L))                                        # [cMpc] the first and last 2 cMpc of the path
n_px = int(np.ceil(P / 0.5)); v_ring = (np.arange(n_px) + 0.5) * P / n_px   # one period [0, P), 0.5 km/s pixels
dtau_p = np.asarray(compute_optical_depth(same, v_ring, lya, per_piece=True, velocity_period=True,
                                          resolution_tolerance=None))[0, 0]   # every d tau_p (N_p, N_v), wrapped
fig, axs, info = plot_deposition_steps(parts, skewer, REGIONS, v_ring, dtau_p, period=P)
fig.suptitle("The two ends of one skewer through TNG50-4 meet at the seam: its pieces on the ring of s, on the ring of v, "
             "and their τ", fontsize=21)
plt.show()
lab = info["piece_labels"]                                                  # {"E3a": index of the piece in the absorbers}
v_par = np.asarray(parts["absorbers"]["HI"].v_los[0])[list(lab.values())]
print(f"end: {info['n_cells']['end']} cells, {info['n_pieces']['end']} pieces; start: {info['n_cells']['start']} cells, "
      f"{info['n_pieces']['start']} pieces; v_par of the labelled pieces {v_par.min():+.0f} to {v_par.max():+.0f} km/s "
      f"({v_par.min() * L / P:.1f}-{v_par.max() * L / P:.1f} cMpc of Hubble flow)")
print(f"on the velocity seam: {info['seam_piece']}, v_c = P {info['seam_v_c']:+.1f} km/s, W_p = {info['seam_W']:.1f} km/s; "
      f"its profile is cut at v_c + P/2 = {info['seam_v_c'] + P / 2:.0f} km/s; all the cuts remove {cut / column:.0e} of the column")

*Reading the figure.* Left, the end of the path; right, its start; the seam between them.

- **(a) The cells**, as in notebook 05, in the colour of their H I density, with their peculiar velocities (yellow
  arrows from the generating points), the skewer (red) with its unit tangent $\hat t$ (black arrows), and the box faces
  where the path ends and starts (green planes). Cell S1 is crossed at both ends: $x = 0 \equiv L$ is one place.
- **(b) The pieces** along the path. Each bar is one piece $p$, a cell inside one dual segment, of height $n_p$ and
  width $\ell_p$. The solid lines are the cell crossings and the dotted ones the ends of the dual segments: together they
  cut the path into the pieces. The yellow arrow of a piece is its $v_\parallel = \mathbf{v}\cdot\hat t$, as
  `absorbers_from_components` computes it, along $s$. Here every one points to larger $s$: the gas moves away from
  the observer (printed above).
- **(c) The ring of $s$.** A crossing of the box closes on itself, so $s$ is periodic over $L$. The bars of (b) become
  blocks on a ring: angle $s$, angular width $\ell_p$, height $n_p$, $v_\parallel$ along the ring (same scale). In
  space the pieces tile the ring and never overlap. The pieces of S1 at the end (S1c, S1d) continue those at the start
  (S1a, S1b) across the seam: one crossing of one cell.
- **(d) The ring of $v_\dagger$.** Each piece moves to its line centre $v_{c,p} = v_H + v_\parallel$, keeping its height and
  label. Its angular width becomes $W_p$, the Hubble flow across it. Without peculiar velocities the angle would hardly
  change ($v_H \simeq (P/L)\,s$); here the whole pattern turns clockwise by $v_\parallel$. Pieces from different places
  now overlap: around 100 km/s, E6–E11 and S1–S3 absorb at the same velocity. E1a (bold) lies on the seam
  ($v_c = P - 1.4$ km/s), and every end-of-box piece after it lands at small $v_\dagger$: its line centre is beyond $P$ (case
  (i) above), and the ring puts it there.
- **(e) From pieces to $\tau$.** Each block becomes its $d\tau_p(v_\dagger)$: a Voigt profile at the piece's temperature, of
  area $\propto n_p \ell_p a_p^{-2}$, with the detuning wrapped to $[-P/2, P/2)$. On the ring the wrap is seamless: E1a's
  profile runs on across $0 \equiv P$ as one curve (case (ii) is the same thing for a profile's tail). Black: the total
  $\tau$, the sum of all pieces, including those outside the labelled stretches (grey in (c) and (d)). The radius grows
  as $\sqrt{\tau}$, so that the weak profiles stay visible.
- **The cut at $P/2$** (insets of (d) and (e)): each profile is evaluated within half a period of its centre only. For E1a
  that is the ray opposite the seam, $v_c + P/2$; the images beyond it are not summed (printed above).

**When to use which.**

- **No wrap** (the default) is the integral along the path as it is: right for any path that is not a single crossing
  of a periodic box (a light cone, a curved path, a segment of a longer sightline), and the cheapest. For a box it
  leaves out the absorption that falls beyond the ends: in 16 TNG50-4 skewers the mean flux is higher by about
  $7\times10^{-3}$ than with a periodic spectrum, and the P1D differs by up to 25 % at $k < 0.1$ s/km.
- **`velocity_period=True`** gives a spectrum periodic over the box, on a grid of one period. FFT-based estimators of a
  box need it (08). Its approximations: wrapped gas keeps its own scale factor (that of the face it belongs to), and
  only the nearest image of each piece is kept, which matters for damped wings in small boxes.

## 2 · Line-centred frames, the global velocity axis and the observed wavelength

These are three axes for the same spectrum; changing from one to another is a change of axis only.

- **The line-centred frame** of every line: each line $j$ is integrated on a grid of $v_\dagger$, its own frame. By notebook
  06, §2, the line centre of the gas at $x$ is the same $v_c(x)$ in every frame: it does not depend on $\lambda_j$
  (`core.spectral.line_centre_velocity_kms`).
- **The global velocity axis** of a reference line (here $\mathrm{Ly}\alpha$): the same observed wavelength sits at
  $v_{\rm Ly\alpha} = v_\dagger - \Delta v_j$ in the reference frame, with

$$
\Delta v_j = c\ln\frac{\lambda_{\rm Ly\alpha}}{\lambda_j}
$$

  (`LineBundle.delta_v`). Going to this axis is a relabelling only: line $j$, computed on its own window $[0, P)$, moves
  to $[-\Delta v_j, P - \Delta v_j)$, and the reference line stays where it is. `compute_optical_depth(...,
  line_frame="reference")` evaluates every line directly on that axis, line $j$ at $v + \Delta v_j$ in its own frame
  (`spectral.velocities_in_line_frames`), over whatever grid it is given: on a grid that holds both lines it would also
  evaluate $\mathrm{Ly}\alpha$ below 0, outside its window (the far wings of the absorbers near the start), so here
  each line is computed on its own window only.
- **The observed wavelength** (`spectral.line_wavelength_grid_A`):

$$
\lambda_{\rm obs} = \lambda_j\, \frac{a_{\rm obs}}{a_\dagger}\, e^{v_\dagger/c} .
$$

**The gas.** The skewer of §1, sampled every km/s (`skewer_fine`), for $\mathrm{Ly}\alpha$ and Si III $\lambda$1206, with a
periodic spectrum (`velocity_period=True`), as a box crossing calls for (§1). Si III lives only in dense, warm gas
around haloes (notebook 05, §2): along this line through the diffuse gas of the box centre its $\tau$ stays below
$10^{-8}$. **For display only, the Si III optical depth is multiplied by a factor `SI_BOOST`** (a power of 10 that brings
its strongest feature to $\tau \approx 1$, $10^8$ here), $F_{\rm SiIII} = e^{-{\rm SI\_BOOST}\,\tau_{\rm SiIII}}$, in all
three panels and labelled in the legends; the computation is unchanged, and no combined flux is drawn. The dotted lines
mark its strongest features.

In [ ]:
from skewerjax.utils.plotting import plot_frames_axes
from skewerjax.core import wrap_to_ref_line
a_dag = float(skewer_fine.a0[0])                                              # the reference scale factor: a at the first point
v_box = np.arange(0.5, P, 1.0)                                               # each line's own window [0, P), 1 km/s pixels
tau_own = np.asarray(compute_optical_depth(skewer_fine, v_box, lines, gas, velocity_period=True, resolution_tolerance=None))[:, 0]
F_own = np.exp(-tau_own)
dV = float(lines.delta_v[1])
peaks = []                                                                    # the three strongest Si III features, 150 km/s apart
for j in np.argsort(tau_own[1])[::-1]:
    if all(abs(v_box[j] - q) > 150 for q in peaks):
        peaks.append(v_box[j])
    if len(peaks) == 3:
        break
SI_BOOST = 15.0 ** np.round(np.log10(1.0 / tau_own[1].max()))                # display only: Si III's strongest feature at tau ~ 1
n_w = int(np.ceil(P)); v_one = (np.arange(n_w) + 0.5) * P / n_w           # (d): a uniform grid of exactly one period
tau_one = np.asarray(compute_optical_depth(skewer_fine, v_one, lines, gas, velocity_period=True, resolution_tolerance=None))
tau_to_ref = np.asarray(wrap_to_ref_line(tau_one, v_one, lines, velocity_period=P))   # every line on the Lya axis, modulo P
fig, axs = plot_frames_axes(v_box, F_own, lines, a_dagger=a_dag, a_obs=1.0, centres=peaks,     # (b): each window relabelled by -Delta v_j
                            tau_scale={"SiIII-1206": SI_BOOST},
                            wrapped=dict(v=v_one, tau=tau_one[:, 0], tau_ref=tau_to_ref[:, 0], period=P))
add_panel_letters(axs)
fig.suptitle(r"The same skewer in each line's own frame (a), on the global $\mathrm{Ly}\alpha$ axis (b), in observed wavelength (c), "
             r"and wrapped to the $\mathrm{Ly}\alpha$ frame (d)",
             fontsize=21)
plt.show()
print(f"<F>: Lya {F_own[0].mean():.3f}, Si III 1 - {1 - F_own[1].mean():.1e} (true tau, max {tau_own[1].max():.1e}); "
      f"drawn with tau x {SI_BOOST:.0e}: max {SI_BOOST * tau_own[1].max():.2f}")
print(f"global axis: Lya on [{v_box[0]:.1f}, {v_box[-1]:.1f}] km/s (its own window), Si III on [{v_box[0] - dV:.1f}, {v_box[-1] - dV:.1f}] km/s")
print(f"Delta v(Si III) = c ln(lambda_Lya / lambda_SiIII) = c ln({lines.lines[0].lambda0_A:.4f} / {lines.lines[1].lambda0_A:.4f}) = "
      f"{phys.c_kms * np.log(lines.lines[0].lambda0_A / lines.lines[1].lambda0_A):+.1f} km/s: the strongest Si III feature, at "
      f"{peaks[0]:.1f} km/s in its own frame, sits at {peaks[0] - dV:.1f} km/s on the Lya axis (bluer: shorter wavelength)")
si_ref = np.asarray(compute_optical_depth(skewer_fine, v_box - dV, LineBundle(["SiIII-1206"], ref_line_id="HI-LyA"), gas.select_species("SiIII"),
                                          line_frame="reference", velocity_period=True, resolution_tolerance=None))[0, 0]
print(f"line_frame='reference' on the relabelled grid gives the same Si III: max |dtau| / max tau = "
      f"{np.abs(si_ref - tau_own[1]).max() / tau_own[1].max():.1e}")
tau_ref_frame = np.asarray(compute_optical_depth(skewer_fine, v_one, lines, gas, line_frame="reference", velocity_period=True,
                                                 resolution_tolerance=None))
print(f"(d) wrap_to_ref_line (interpolated) vs line_frame='reference' (evaluated) on one period: Si III max |dtau| / max tau = "
      f"{np.abs(tau_to_ref[1, 0] - tau_ref_frame[1, 0]).max() / tau_ref_frame[1, 0].max():.1e}; "
      f"Lya unchanged: {np.array_equal(tau_to_ref[0, 0], tau_one[0, 0])}")


*Reading the figure.*

- (a) In its own frame each line absorbs where the gas is: the same gas at the same $v_\dagger$ in both lines. Si III
  (its $\tau$ multiplied by `SI_BOOST` for display) absorbs only where the gas is densest, under saturated
  $\mathrm{Ly}\alpha$ absorbers (dotted: its strongest features).
- (b) On the $\mathrm{Ly}\alpha$ axis, each line keeps exactly the window it was computed on (dash-dotted edges, in its
  colour). $\mathrm{Ly}\alpha$ stays on $[0, P)$; Si III moves by $-\Delta v_j = -2270$ km/s, to the blue, because its
  rest wavelength is shorter: the same gas absorbs Si III at a shorter observed wavelength, 2270 km/s bluer on the
  $\mathrm{Ly}\alpha$ axis.
- (c) On the observed-wavelength axis the two lines are $9.2\ \text{Å} \times (a_{\rm obs}/a_\dagger)$ apart, and they overlap:
  a Si III feature falls into the $\mathrm{Ly}\alpha$ forest of gas 2270 km/s closer to the observer.
- (d) `wrap_to_ref_line` does (b) after the computation: $\tau_j^{\rm ref}(v) = \tau_j(v + \Delta v_j)$ by linear
  interpolation, and for a periodic spectrum (`velocity_period=P`, on a uniform grid of exactly one period) modulo $P$.
  So Si III does not leave the window as in (b): it turns by $-\Delta v_j$ around the ring of §1 and stays on $[0, P)$,
  dashed (own frame) to solid (the $\mathrm{Ly}\alpha$ frame), as `line_frame="reference"` would evaluate it (printed above).